# 04 · Мастер-датасет

Единая точка входа для EDA и модели. Ноутбук собирает признаки из всех этапов в одну таблицу кандидатов. Объекты 2ГИС и районы хранятся в отдельных справочниках. Код сборки — в `notebooks/master_dataset.py`.

**Почему не одна «плоская» таблица на всё.** У данных разная гранулярность:
- кандидат (2 822 точки);
- объект 2ГИС (тысячи заведений);
- район (8).

Если склеить их в одну таблицу, строки размножатся или данные потеряются. Поэтому таблиц три, и связаны они ключами:

| таблица | одна строка = | ключ | откуда |
|---|---|---|---|
| `candidates_master.parquet` | кандидат | `candidate_id` | 02 (сетка и базовые признаки), 03_osm, 03_2gis_all_coffee, 03_2gis_demand_layers, районы |
| `poi_2gis.parquet` | объект 2ГИС | `id` (текст) | сырые ответы всех сборщиков 2ГИС; полный JSON в `raw_json` |
| `districts.parquet` | район | `district` | границы районов + statgov |

**Гарантии без потерь.**
1. Каждый источник присоединяется к сетке один к одному. Проверяется, что id уникальны и что нет ни лишних, ни потерянных кандидатов. x/y источника должны совпадать с геометрией сетки до 1 см; источник без `candidate_id` сопоставляется по координатам.
2. Если два источника дают столбец с одним именем, сборка останавливается с ошибкой: тихих переименований нет.
3. После сборки каждый столбец каждого источника сравнивается с исходным: значения, пропуски, типы.
4. Технические столбцы в таблицу кандидатов не переносятся, но остаются в справочниках. Каждый такой случай записан в `manifest.json`.
5. В `manifest.json` есть sha256 и число строк каждого входа и выхода. По нему видно, на каких данных собран датасет.

**Чего пока не хватает на этом компьютере.**
- Жильё, школы и границы районов собираются сами в `03_residential_schools_districts.ipynb`.
- Нет `almaty_features.parquet` из 02: его подготовленные слои OSM есть только у Давида.
- Нет `almaty_district_stats.csv` из statgov: нужны Excel-файлы stat.gov.kz.

Без них сборка проходит, а их отсутствие записывается в отчёт. Когда файлы появятся в `data/processed/`, достаточно перезапустить ноутбук.

In [1]:
import json
from pathlib import Path
import sys

import geopandas as gpd
import numpy as np
import pandas as pd

PROJECT_ROOT = (Path("..") if Path.cwd().name == "notebooks" else Path(".")).resolve()
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import master_dataset as md_
from master_dataset import (FEATURE_SOURCES, FeatureSource, MasterBuildError, assert_lossless, assign_districts,
                            build_district_table, build_master, build_poi_2gis, check_split_copies, load_master)

pd.set_option("display.max_columns", 80)
for s in FEATURE_SOURCES:
    print(f"{s.name:12s} {'есть' if (PROJECT_ROOT / s.path).is_file() else 'НЕТ '}  {s.path}  ← {s.owner}")

base02       есть  data/processed/almaty_features.parquet  ← 02_candidate_locations
osm          есть  data/processed/features/osm_commercial_features.csv  ← 03_osm_commercial_features
osm_context  НЕТ   data/processed/features/osm_residential_schools.csv  ← 03_residential_schools_districts
dgis_coffee  НЕТ   data/processed/features/dgis_coffee_features.csv  ← 03_2gis_all_coffee
dgis_demand  НЕТ   data/processed/features/dgis_demand_features.csv  ← 03_2gis_demand_layers


## Офлайн-тесты сборки

Тесты собирают маленький искусственный проект и проверяют:
- что сохраняются значения и типы, включая пропуски, `Int64` и `bool`;
- что ловятся расхождения x/y, лишние и потерянные id, одинаковые имена столбцов;
- назначение района внутри полигона, на границе и по ближайшему, с разным написанием названий;
- справочник объектов 2ГИС из сырых ответов;
- что отсутствующий источник попадает в отчёт.

In [2]:
import sqlite3
import tempfile
import unittest
from contextlib import closing

import shapely

import dgis_api


def make_project(tmp, n=6):
    root = Path(tmp)
    (root / "data/processed/features").mkdir(parents=True)
    xs = 650_000.0 + 500 * np.arange(n)
    grid = gpd.GeoDataFrame({"candidate_id": np.arange(n, dtype="int64")},
                            geometry=gpd.points_from_xy(xs, np.full(n, 4_785_000.0)), crs="EPSG:32643")
    grid.to_file(root / "data/processed/candidate_locations.gpkg", driver="GPKG")
    boundary = gpd.GeoDataFrame(geometry=[shapely.box(xs[0] - 100, 4_784_900, xs[-1] + 100, 4_785_100)], crs="EPSG:32643")
    boundary.to_file(root / "data/processed/almaty_boundary.gpkg", driver="GPKG")
    feats = pd.DataFrame({"candidate_id": grid["candidate_id"], "x": grid.geometry.x, "y": grid.geometry.y,
                          "shops_500m": [3, 0, 1, 2, 5, 0], "metro_dist": [100.5, np.nan, 300.0, 1.0, 2.0, 3.0],
                          "flag": [True, False, True, True, False, True]}).sample(frac=1, random_state=1)
    feats.to_csv(root / "data/processed/features/a.csv", index=False)
    geo = grid.sample(frac=1, random_state=2).drop(columns="candidate_id").reset_index(drop=True)
    geo["cafes_500m"] = [7, 1, 0, 2, 4, 3]
    geo.to_parquet(root / "data/processed/base.parquet")
    return root, grid


SOURCES = (FeatureSource("a", "data/processed/features/a.csv", "test_a"),
           FeatureSource("base", "data/processed/base.parquet", "test_base", key="geometry"))


class MasterTests(unittest.TestCase):
    def setUp(self):
        temp = tempfile.TemporaryDirectory()
        self.addCleanup(temp.cleanup)
        self.root, self.grid = make_project(temp.name)

    def build(self, sources=SOURCES, **kw):
        return build_master(self.root, sources=sources, expected=6, with_poi=False, **kw)

    def test_lossless_roundtrip_and_geometry_join(self):
        result = self.build()
        master = result["master"]
        self.assertEqual(len(master), 6)
        a = pd.read_csv(self.root / "data/processed/features/a.csv").set_index("candidate_id").sort_index()
        pd.testing.assert_series_equal(master.set_index("candidate_id")["metro_dist"], a["metro_dist"], check_names=False)
        geo = gpd.read_parquet(self.root / "data/processed/base.parquet")
        by_x = dict(zip(geo.geometry.x.round(3), geo["cafes_500m"]))
        self.assertEqual(master["cafes_500m"].tolist(), [by_x[round(x, 3)] for x in master["x"]])
        back = gpd.read_parquet(self.root / "data/processed/master/candidates_master.parquet")
        pd.testing.assert_frame_equal(pd.DataFrame(back.drop(columns="geometry")),
                                      pd.DataFrame(master.drop(columns="geometry")))
        manifest = json.loads((self.root / "data/processed/master/manifest.json").read_text(encoding="utf-8"))
        self.assertIn("district", manifest["skipped_sources"])
        self.assertEqual(manifest["outputs"]["candidates_master.parquet"]["rows"], 6)
        self.assertEqual(set(result["dictionary"]["column"]),
                         {"candidate_id", "x", "y", "lon", "lat", "shops_500m", "metro_dist", "flag", "cafes_500m"})

    def test_xy_mismatch_is_rejected(self):
        a = pd.read_csv(self.root / "data/processed/features/a.csv")
        a.loc[0, "x"] += 0.02
        a.to_csv(self.root / "data/processed/features/a.csv", index=False)
        with self.assertRaisesRegex(MasterBuildError, "x/y"):
            self.build()

    def test_missing_and_extra_ids_are_rejected(self):
        a = pd.read_csv(self.root / "data/processed/features/a.csv")
        for broken in (a.iloc[1:], pd.concat([a, a.iloc[[0]].assign(candidate_id=99)]), pd.concat([a, a.iloc[[0]]])):
            broken.to_csv(self.root / "data/processed/features/a.csv", index=False)
            with self.assertRaises(MasterBuildError):
                self.build()

    def test_column_collision_is_rejected(self):
        twin = FeatureSource("twin", "data/processed/features/a.csv", "test_twin")
        with self.assertRaisesRegex(MasterBuildError, "Одинаковые столбцы"):
            self.build(sources=SOURCES + (twin,))

    def test_changed_value_is_detected(self):
        result = self.build(write=False)
        frames = {"a": pd.read_csv(self.root / "data/processed/features/a.csv").set_index("candidate_id").drop(columns=["x", "y"])}
        broken = result["master"].copy()
        broken.loc[2, "shops_500m"] = 999
        with self.assertRaises(MasterBuildError):
            assert_lossless(broken, frames)

    def test_missing_source_is_reported(self):
        ghost = FeatureSource("ghost", "data/processed/features/ghost.csv", "test_ghost")
        result = self.build(sources=SOURCES + (ghost,), write=False)
        self.assertIn("ghost", result["skipped"])

    def test_districts_within_boundary_nearest_and_names(self):
        x0 = 650_000.0
        districts = gpd.GeoDataFrame({"NAME": ["Ауэзовский ", "Медеуский"], "OBJECTID": [1, 2], "RAION": ["312", "315"]},
                                     geometry=[shapely.box(x0 - 250, 4_784_000, x0 + 1000, 4_786_000),
                                               shapely.box(x0 + 1000, 4_784_000, x0 + 1600, 4_786_000)], crs="EPSG:32643")
        stats = pd.DataFrame({"district": ["Ауезовский район", "Медеуский"], "population_2026": [362597, 257104]})
        assigned = assign_districts(self.grid, districts)
        self.assertEqual(assigned["district"].tolist(),
                         ["Ауэзовский", "Ауэзовский", "Ауэзовский", "Медеуский", "Медеуский", "Медеуский"])
        self.assertEqual(assigned["district_method"].tolist(), ["within", "within", "boundary", "within", "nearest", "nearest"])
        self.assertEqual(assigned["district_distance_m"].round(1).tolist(), [0, 0, 0, 0, 400.0, 900.0])
        dim = build_district_table(districts, stats)
        self.assertEqual(dim["population_2026"].tolist(), [362597, 257104])
        with self.assertRaisesRegex(MasterBuildError, "не совпали"):
            build_district_table(districts, stats.assign(district=["Алмалинский", "Медеуский"]))
        districts.to_parquet(self.root / "data/processed/almaty_districts.parquet")
        stats.to_csv(self.root / "data/processed/almaty_district_stats.csv", index=False)
        master = self.build(write=False)["master"]
        self.assertEqual(master["district_population_2026"].tolist(), [362597] * 3 + [257104] * 3)
        self.assertNotIn("district_OBJECTID", master.columns)
        self.assertIn("district_RAION", master.columns)
        # без статистики: районы и атрибуты границ подключаются, отсутствие статистики — в отчёте
        (self.root / "data/processed/almaty_district_stats.csv").unlink()
        only_bounds = self.build(write=False)
        self.assertIn("district_stats", only_bounds["skipped"])
        self.assertEqual(only_bounds["master"]["district_RAION"].tolist(), ["312"] * 3 + ["315"] * 3)
        self.assertNotIn("district_population_2026", only_bounds["master"].columns)

    def test_poi_from_raw_checkpoints(self):
        coffee = self.root / dgis_api_paths["coffee"]
        coffee.parent.mkdir(parents=True)
        with closing(sqlite3.connect(coffee)) as db:
            db.executescript(dgis_api.AREA_SCHEMA)
            db.execute("INSERT INTO run (id, plan) VALUES (1, '{}')")
            items = [{"id": "701", "name": "Кофейня 1", "point": {"lon": 76.9, "lat": 43.2},
                      "rubrics": [{"id": "162", "name": "Кофейни", "kind": "primary"}],
                      "attribute_groups": [{"attributes": [{"tag": "food_service_avg_price", "name": "Чек 2500 ₸"}]}]},
                     {"id": "702_hash", "name": "Точка", "point": {"lon": 76.91, "lat": 43.2},
                      "rubrics": [{"id": "58871", "name": "Точки кофе", "kind": "primary"}]}]
            db.executemany("INSERT INTO items VALUES (?, ?)", [(dgis_api.base_id(i["id"]), json.dumps(i)) for i in items])
            db.commit()
        gap = self.root / dgis_api_search["gap"]
        gap.parent.mkdir(parents=True)
        with closing(sqlite3.connect(gap)) as db:
            db.executescript(dgis_api.SCHEMA)
            db.execute("INSERT INTO run (id, plan) VALUES (1, '{}')")
            db.executemany("INSERT INTO organizations VALUES (?, ?, ?, ?, ?, ?)",
                           [("701", "Кофейня 1", 76.9, 43.2, 4.5, 10), ("703", "Кафе из поиска", 76.92, 43.2, 4.0, 3)])
            db.commit()
        poi, used = build_poi_2gis(self.root, area=shapely.box(0, 0, 1e7, 1e7))
        self.assertEqual(poi["id"].tolist(), ["701", "702", "703"])
        self.assertEqual(poi["sources"].tolist(), ["coffee;gap", "coffee", "gap"])
        self.assertEqual(poi["is_coffee_shop"].tolist(), [True, False, False])
        self.assertEqual(poi["is_coffee_point"].tolist(), [False, True, False])
        self.assertEqual(poi.loc[0, "avg_check_kzt"], 2500)
        self.assertTrue(pd.isna(poi.loc[2, "raw_json"]))   # объект только из текстового поиска — JSON нет
        self.assertEqual(json.loads(poi.loc[1, "raw_json"])["id"], "702_hash")
        poi.to_parquet(self.root / "poi.parquet")
        back = gpd.read_parquet(self.root / "poi.parquet")
        self.assertEqual(back["id"].tolist(), poi["id"].tolist())


dgis_api_paths, dgis_api_search = md_.DGIS_AREA_CHECKPOINTS, md_.DGIS_SEARCH_CHECKPOINTS
suite = unittest.defaultTestLoader.loadTestsFromTestCase(MasterTests)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
if not test_result.wasSuccessful():
    raise AssertionError("Проверки сборки мастер-датасета завершились ошибкой.")

test_changed_value_is_detected (__main__.MasterTests.test_changed_value_is_detected) ... ERROR
test_column_collision_is_rejected (__main__.MasterTests.test_column_collision_is_rejected) ... ERROR
test_districts_within_boundary_nearest_and_names (__main__.MasterTests.test_districts_within_boundary_nearest_and_names) ... ERROR
test_lossless_roundtrip_and_geometry_join (__main__.MasterTests.test_lossless_roundtrip_and_geometry_join) ... ERROR
test_missing_and_extra_ids_are_rejected (__main__.MasterTests.test_missing_and_extra_ids_are_rejected) ... ERROR
test_missing_source_is_reported (__main__.MasterTests.test_missing_source_is_reported) ... ERROR
test_poi_from_raw_checkpoints (__main__.MasterTests.test_poi_from_raw_checkpoints) ... ok
test_xy_mismatch_is_rejected (__main__.MasterTests.test_xy_mismatch_is_rejected) ... ERROR

ERROR: test_changed_value_is_detected (__main__.MasterTests.test_changed_value_is_detected)
----------------------------------------------------------------------
T

AssertionError: Проверки сборки мастер-датасета завершились ошибкой.

## Сборка

Отдельные CSV признаков OSM (`shops_500m.csv` и т. п.) должны совпадать со сводным `osm_commercial_features.csv`. Это проверяется перед сборкой. В мастер-таблицу входит только сводный файл, чтобы не было двух копий одних данных.

In [ ]:
print("Отдельные файлы OSM совпадают со сводным:", check_split_copies(PROJECT_ROOT, "data/processed/features/osm_commercial_features.csv"))
result = build_master(PROJECT_ROOT)
master, dictionary, poi = result["master"], result["dictionary"], result["poi"]
print(f"\nМастер-таблица: {master.shape[0]} кандидатов × {master.shape[1]} столбцов")
for r in result["reports"]:
    print(f"  {r['name']:12s} ключ={r['key']:12s} строк={r['rows']}  признаков={len(r['columns'])}  "
          f"x/y до {r.get('xy_max_diff_m', 0):.4f} м  не перенесено: {list(r['dropped'])}")
print("Пропущенные источники:")
for name, reason in result["skipped"].items():
    print(f"  {name}: {reason}")
print(f"\nОбъекты 2ГИС: {len(poi)} (внутри города + 1 км: {int(poi['in_area'].sum())}); источники: {result['poi_sources']}")
print("Выходы:", json.dumps(result["outputs"], ensure_ascii=False, indent=1))

Отдельные файлы OSM совпадают со сводным: ['attractions_1000m.csv', 'major_roads_distance.csv', 'malls_1000m.csv', 'metro_dist.csv', 'parking_500m.csv', 'restaurants_500m.csv', 'shops_500m.csv', 'transit_stops_500m.csv']


MasterBuildError: В candidate_locations.gpkg нет candidate_id — пересоберите сетку через 02.

## Словарь данных и профиль

Словарь — `data/processed/master/data_dictionary.csv`. Для каждого столбца там указаны источник, ноутбук-владелец, тип, число пропусков и описание.

In [ ]:
display(dictionary)
print("Столбцы с пропусками:", dictionary.loc[dictionary["null"] > 0, ["column", "null", "description"]].to_dict("records"))

,column,source,owner,dtype,non_null,null,description
0,candidate_id,base,02_candidate_locations,int64,2822,0,"ID точки-кандидата (из 02, сетка 500 м)"
1,x,base,02_candidate_locations,float64,2822,0,"Координата X, м (EPSG:32643)"
2,y,base,02_candidate_locations,float64,2822,0,"Координата Y, м (EPSG:32643)"
3,lon,base,02_candidate_locations,float64,2822,0,Долгота (WGS84)
4,lat,base,02_candidate_locations,float64,2822,0,Широта (WGS84)
5,shops_500m,osm,03_osm_commercial_features,int64,2822,0,Магазины OSM в 500 м (после склейки дублей)
6,malls_1000m,osm,03_osm_commercial_features,int64,2822,0,ТРЦ OSM в 1000 м
7,restaurants_500m,osm,03_osm_commercial_features,int64,2822,0,Рестораны/кафе/фастфуд OSM в 500 м
8,metro_dist,osm,03_osm_commercial_features,float64,2822,0,"Расстояние до станции метро, м"
9,transit_stops_500m,osm,03_osm_commercial_features,int64,2822,0,Остановки ОТ в 500 м (без дублей)


Столбцы с пропусками: [{'column': 'dgis_coffee_rating_mean_500m', 'null': 2202, 'description': 'Средний рейтинг кофеен в 500 м'}, {'column': 'dgis_coffee_avg_check_median_500m', 'null': 2293, 'description': 'Медианный средний чек кофеен в 500 м, ₸'}]


In [ ]:
layer_cols = [c for c in poi.columns if c.startswith("is_")]
inside = poi[poi["in_area"]]
display(inside[layer_cols].sum().rename("объектов в городе + 1 км").to_frame())
print("Объектов сразу в нескольких слоях:", int((inside[layer_cols].sum(axis=1) > 1).sum()))

,объектов в городе + 1 км
is_coffee_shop,971
is_coffee_point,490
is_business_center,308
is_coworking,70
is_bakery,293
is_cafe_confectionery,243
is_fitness,449
is_education,268
is_mall,195
is_coffee_vending,485


Объектов сразу в нескольких слоях: 258


## Как пользоваться

```python
from master_dataset import load_master
master = load_master()                      # GeoDataFrame: 2 822 кандидата × все признаки, EPSG:32643
X = load_master(with_geometry=False)        # то же без геометрии — для модели

import geopandas as gpd
poi = gpd.read_parquet("../data/processed/master/poi_2gis.parquet")   # объекты 2ГИС, id — текст
```

Мастер-таблицу не редактируйте вручную. Новый признак добавляется так:
1. ноутбук этапа 03 пишет CSV с `candidate_id, x, y, <признаки>`;
2. в `FEATURE_SOURCES` в `master_dataset.py` добавляется одна строка;
3. ноутбук 04 перезапускается.

In [ ]:
from master_dataset import build_master

result = build_master(
    "..",
    write=False,
    with_poi=False
)

master = result["master"]

print("Shape:", master.shape)

print("\nLoaded sources:")
for report in result["reports"]:
    print(report["name"], len(report["columns"]))

print("\nSkipped sources:")
print(result["skipped"])

print("\nDistrict methods:")
print(master["district_method"].value_counts(dropna=False))

print("\nColumns:")
print(master.columns.tolist())

MasterBuildError: В candidate_locations.gpkg нет candidate_id — пересоберите сетку через 02.

In [ ]:
import geopandas as gpd

grid = gpd.read_file("../data/processed/candidate_locations.gpkg")

print("Shape:", grid.shape)
print("CRS:", grid.crs)

print("\nColumns:")
print(grid.columns.tolist())

print("\nFirst rows:")
print(grid.head())

print("\nIndex:")
print(grid.index[:10])

Shape: (2822, 14)
CRS: EPSG:32643

Columns:
['cafes_500m', 'cafes_300m', 'cafes_1000m', 'bus_stops_500m', 'residential_buildings_500m', 'houses_500m', 'apartments_500m', 'residential_unknown_500m', 'office_polygons_500m', 'offices_pois_500m', 'universities_500m', 'colleges_500m', 'schools_500m', 'geometry']

First rows:
   cafes_500m  cafes_300m  cafes_1000m  bus_stops_500m  \
0           0           0            0               0   
1           0           0            0               0   
2           0           0            0               0   
3           0           0            0               0   
4           0           0            0               0   

   residential_buildings_500m  houses_500m  apartments_500m  \
0                           0            0                0   
1                           0            0                0   
2                           0            0                0   
3                           0            0                0   
4             

In [ ]:
import pandas as pd
import numpy as np

commercial = pd.read_csv(
    "../data/processed/features/osm_commercial_features.csv"
)

commercial = commercial.sort_values("candidate_id").reset_index(drop=True)

distances = np.hypot(
    grid.geometry.x.to_numpy() - commercial["x"].to_numpy(),
    grid.geometry.y.to_numpy() - commercial["y"].to_numpy()
)

print("Grid rows:", len(grid))
print("CSV rows:", len(commercial))
print("Unique IDs:", commercial["candidate_id"].is_unique)

print(
    "IDs from 0 to 2821:",
    commercial["candidate_id"].tolist() == list(range(len(grid)))
)

print("Maximum distance (m):", distances.max())
print("Mismatches > 0.01 m:", (distances > 0.01).sum())

Grid rows: 2822
CSV rows: 2822
Unique IDs: True
IDs from 0 to 2821: True
Maximum distance (m): 0.003219610343181565
Mismatches > 0.01 m: 0


In [ ]:
grid_fixed = grid.copy()

grid_fixed["candidate_id"] = commercial["candidate_id"].to_numpy()

grid_fixed.to_file(
    "../data/processed/candidate_locations_with_ids.gpkg",
    driver="GPKG"
)

In [ ]:
fixed = gpd.read_file(
    "../data/processed/candidate_locations_with_ids.gpkg"
)

print("Shape:", fixed.shape)
print("CRS:", fixed.crs)
print("Unique IDs:", fixed["candidate_id"].is_unique)
print("Missing IDs:", fixed["candidate_id"].isna().sum())
print("Geometry unchanged:", fixed.geometry.equals(grid.geometry))

Shape: (2822, 15)
CRS: EPSG:32643
Unique IDs: True
Missing IDs: 0
Geometry unchanged: True


In [ ]:
from master_dataset import build_master

result = build_master(
    "..",
    write=False,
    with_poi=False
)

print("Shape:", result["master"].shape)

print("\nLoaded sources:")
for report in result["reports"]:
    print(report["name"], report["columns"])

print("\nSkipped sources:")
print(result["skipped"])

print("\nDistrict methods:")
print(result["master"]["district_method"].value_counts(dropna=False))

In [ ]:
from master_dataset import load_candidates

candidates = load_candidates("..")

print("Shape:", candidates.shape)
print("Unique IDs:", candidates["candidate_id"].is_unique)
print("CRS:", candidates.crs)
print("ID range:", candidates["candidate_id"].min(), candidates["candidate_id"].max())

Shape: (2822, 6)
Unique IDs: True
CRS: EPSG:32643
ID range: 0 2821


In [ ]:
from master_dataset import build_master

result = build_master(
    "..",
    write=False,
    with_poi=False
)

master = result["master"]

print("Master shape:", master.shape)

print("\nLoaded sources:")
for report in result["reports"]:
    print(report["name"], len(report["columns"]))

print("\nSkipped sources:")
for name, reason in result["skipped"].items():
    print(name, ":", reason)

print("\nDistrict methods:")
print(master["district_method"].value_counts(dropna=False))

Master shape: (2822, 43)

Loaded sources:
base02 13
osm 8

Skipped sources:
osm_context : нет файла data/processed/features/osm_residential_schools.csv (создаёт 03_residential_schools_districts)
dgis_coffee : нет файла data/processed/features/dgis_coffee_features.csv (создаёт 03_2gis_all_coffee)
dgis_demand : нет файла data/processed/features/dgis_demand_features.csv (создаёт 03_2gis_demand_layers)

District methods:
district_method
within     2808
nearest      14
Name: count, dtype: int64


In [ ]:
import geopandas as gpd

base = gpd.read_parquet('../data/processed/almaty_districts.parquet')
enriched = gpd.read_parquet('../data/processed/almaty_districts_with_stats.parquet')

print("BASE:", base.shape)
print("ENRICHED:", enriched.shape)

print(f'Columns only in enriched:{sorted(set(enriched.columns) - set(base.columns))}')
print(f'Columns only in base:{sorted(set(base.columns) - set(enriched.columns))}')

print("\nDistrict names in base:")
print(base["NAME"].tolist())

print("\nDistrict names in enriched:")
print(enriched["NAME"].tolist())

print("\nEnriched missing values:")
print(enriched.isna().sum()[lambda x: x > 0])


BASE: (8, 10)
ENRICHED: (8, 19)
Columns only in enriched:['active_sme_2026', 'avg_salary_kzt', 'district', 'housing_completed_m2', 'housing_investment_mln_kzt', 'housing_investment_per_capita_kzt', 'housing_m2_per_1000_residents', 'population_2026', 'sme_per_1000_residents']
Columns only in base:[]

District names in base:
['Ауэзовский', 'Алатауский', 'Медеуский', 'Наурызбайский', 'Жетысуский', 'Алмалинский', 'Турксибский', 'Бостандыкский']

District names in enriched:
['Ауэзовский', 'Алатауский', 'Медеуский', 'Наурызбайский', 'Жетысуский', 'Алмалинский', 'Турксибский', 'Бостандыкский']

Enriched missing values:
Series([], dtype: int64)


In [ ]:
import geopandas as gpd

from master_dataset import (load_candidates, build_district_table, assign_districts, district_features)

candidates = load_candidates("..")
enriched = gpd.read_parquet('../data/processed/almaty_districts_with_stats.parquet')

district_dim = build_district_table(enriched, stats=None)
assigned = assign_districts(candidates, district_dim)
district_data = district_features(assigned, district_dim)
print(f'Shape: {district_data.shape}')


print("\nDistrict methods:")
print(district_data["district_method"].value_counts())

print("\nColumns:")
print(district_data.columns.tolist())

print("\nMissing values:")
print(district_data.isna().sum()[lambda x: x > 0])

Shape: (2822, 15)

District methods:
district_method
within     2808
nearest      14
Name: count, dtype: int64

Columns:
['district', 'district_method', 'district_distance_m', 'district_RAION', 'district_Population_est', 'district_Schools_capacity', 'district_Students_to_population', 'district_housing_completed_m2', 'district_housing_investment_mln_kzt', 'district_population_2026', 'district_avg_salary_kzt', 'district_active_sme_2026', 'district_sme_per_1000_residents', 'district_housing_m2_per_1000_residents', 'district_housing_investment_per_capita_kzt']

Missing values:
Series([], dtype: int64)


In [3]:
from master_dataset import build_master

result = build_master(
    "..",
    write=False,
    with_poi=False
)

master = result["master"]

print("Shape:", master.shape)

print("\nDistrict statistics:")
print([
    col for col in master.columns
    if col.startswith("district_")
])

print("\nMissing values:")
print(master.isna().sum()[lambda x: x > 0])

print("\nSkipped sources:")
print(result["skipped"])

Shape: (2822, 42)

District statistics:
['district_method', 'district_distance_m', 'district_RAION', 'district_Population_est', 'district_Schools_capacity', 'district_Students_to_population', 'district_housing_completed_m2', 'district_housing_investment_mln_kzt', 'district_population_2026', 'district_avg_salary_kzt', 'district_active_sme_2026', 'district_sme_per_1000_residents', 'district_housing_m2_per_1000_residents', 'district_housing_investment_per_capita_kzt']

Missing values:
Series([], dtype: int64)

Skipped sources:
{'osm_context': 'нет файла data/processed/features/osm_residential_schools.csv (создаёт 03_residential_schools_districts)', 'dgis_coffee': 'нет файла data/processed/features/dgis_coffee_features.csv (создаёт 03_2gis_all_coffee)', 'dgis_demand': 'нет файла data/processed/features/dgis_demand_features.csv (создаёт 03_2gis_demand_layers)'}


In [4]:
print("Loaded sources:")
for report in result["reports"]:
    print(report["name"], len(report["columns"]))

print("\nAll columns:")
for col in master.columns:
    print(col)

print("\nUnique candidates:")
print(master["candidate_id"].is_unique)

print("\nDistrict assignment:")
print(master["district_method"].value_counts())

Loaded sources:
base02 13
osm 8

All columns:
candidate_id
x
y
lon
lat
geometry
cafes_500m
cafes_300m
cafes_1000m
bus_stops_500m
residential_buildings_500m
houses_500m
apartments_500m
residential_unknown_500m
office_polygons_500m
offices_pois_500m
universities_500m
colleges_500m
schools_500m
shops_500m
malls_1000m
restaurants_500m
metro_dist
transit_stops_500m
parking_500m
major_roads_distance
attractions_1000m
district
district_method
district_distance_m
district_RAION
district_Population_est
district_Schools_capacity
district_Students_to_population
district_housing_completed_m2
district_housing_investment_mln_kzt
district_population_2026
district_avg_salary_kzt
district_active_sme_2026
district_sme_per_1000_residents
district_housing_m2_per_1000_residents
district_housing_investment_per_capita_kzt

Unique candidates:
True

District assignment:
district_method
within     2808
nearest      14
Name: count, dtype: int64


In [5]:
import pandas as pd

coffee = pd.read_parquet(
    "../data/processed/2gis_almaty_coffee_extended.parquet"
)

print("Shape:", coffee.shape)

print("\nColumns:")
print(coffee.columns.tolist())

print("\nMissing values:")
print(coffee.isna().sum()[lambda x: x > 0])

Shape: (1251, 5)

Columns:
['id', 'name', 'general_rating', 'general_review_count', 'geometry']

Missing values:
general_rating          47
general_review_count    47
dtype: int64


In [6]:
import geopandas as gpd

coffee_gdf = gpd.read_parquet(
    "../data/processed/2gis_almaty_coffee_extended.parquet"
)

print("CRS:", coffee_gdf.crs)

print("\nGeometry types:")
print(coffee_gdf.geom_type.value_counts(dropna=False))

print("\nMissing geometries:")
print(coffee_gdf.geometry.isna().sum())

print("\nUnique IDs:")
print(coffee_gdf["id"].is_unique)

print("\nDuplicate coordinates:")
print(coffee_gdf.geometry.to_wkb().duplicated().sum())

print("\nBounds:")
print(coffee_gdf.total_bounds)

CRS: {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "ProjectedCRS", "name": "WGS 84 / UTM zone 43N", "base_crs": {"type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviat

In [7]:
coffee_gdf = coffee_gdf.copy()

coffee_gdf["coord_key"] = coffee_gdf.geometry.to_wkb()

duplicates = coffee_gdf[
    coffee_gdf.duplicated("coord_key", keep=False)
].copy()

print("Rows with shared coordinates:", len(duplicates))

print(
    "Unique shared locations:",
    duplicates["coord_key"].nunique()
)

print(
    duplicates[
        ["id", "name", "general_rating", "general_review_count"]
    ].head(20).to_string(index=False)
)

Rows with shared coordinates: 53
Unique shared locations: 24
               id                            name  general_rating  general_review_count
70000001029221435   Urbo Coffee, экспресс-кофейня             4.8                 623.0
70000001029492099         Madison Coffee, кофейня             5.0                  46.0
70000001033452440     The kitchen corner, кофейня             4.0                  12.0
70000001034785233              Starbucks, кофейня             4.8                 344.0
70000001035432714  Maui, ресторан доступной кухни             4.9                1689.0
70000001036440138          Chicken star, арт-кафе             4.9                 194.0
70000001036440155 Flask specialty coffee, кофейня             4.9                 408.0
70000001036466819              Big Apple, кофейня             4.7                 206.0
70000001037469560              Starbucks, кофейня             4.9                 354.0
70000001040975093              Big Apple, кофейня          

In [8]:
groups = (
    duplicates.groupby("coord_key")
    .agg(
        count=("id", "size"),
        names=("name", lambda x: " | ".join(x))
    )
    .sort_values("count", ascending=False)
)

print(groups[["count", "names"]].to_string(index=False))

 count                                                                                                                                                     names
     4 Azkoyen, автомат по продаже снеков | Almaty Vending Company, кофейня самообслуживания | Novara, аппарат по продаже кофе | Novara, аппарат по продаже кофе
     3                                                                                             Alma Tea, кофейня | Wake up coffee, кофейня | Skaski, кофейня
     3                                                                      Urbo Coffee, экспресс-кофейня | Starbucks, кофейня | Kunde Cafe, инклюзивная кофейня
     3                                                                                           Big Apple, кофейня | Love is coffee, кофейня | Drinkit, кофейня
     2                                                                                                                   Big Apple, кофейня | Starbucks, кофейня
     2                            

In [10]:
# 1. Создаём буферы радиусом 500 м
buffers = master[['candidate_id', 'geometry']].copy()
buffers['geometry'] = buffers.geometry.buffer(500)

# находим кофейни внутри каждого буффера
joined = gpd.sjoin(
    buffers,
    coffee_gdf[['id', 'geometry']],
    how='left',
    predicate='intersects'
)

# Считаем уникальные ID кофеен для каждого кандидата
coffee_counts = (joined.groupby('candidate_id')['id'].nunique())

#Добавляем признак в Master Dataset

master['dgis_coffee_sample_500m'] = (
    master['candidate_id'].map(coffee_counts).fillna(0).astype(int)
)

# Проверяем результат
print(master["dgis_coffee_sample_500m"].describe())
print("Zero competitors:", (master["dgis_coffee_sample_500m"] == 0).sum())
print("Missing:", master["dgis_coffee_sample_500m"].isna().sum())


count    2822.000000
mean        1.376329
std         4.026249
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max        36.000000
Name: dgis_coffee_sample_500m, dtype: float64
Zero competitors: 2029
Missing: 0


In [12]:
top_locations = master.nlargest(
    10, 'dgis_coffee_sample_500m'
)

print(
    top_locations[
        [
            'candidate_id',
            'district',
            'cafes_500m',
            'restaurants_500m',
            'dgis_coffee_sample_500m'
        ]
    ].to_string(index=False)
)

 candidate_id      district  cafes_500m  restaurants_500m  dgis_coffee_sample_500m
         1347   Алмалинский          42                21                       36
         1216   Алмалинский          15                10                       35
         1278   Алмалинский          32                23                       33
         1281   Алмалинский          23                16                       33
         1476     Медеуский          20                24                       33
          844 Бостандыкский          25                11                       32
         1345   Алмалинский          38                33                       31
         1344   Алмалинский          36                25                       30
         1346   Алмалинский          34                23                       30
         1409 Бостандыкский          22                12                       30


In [13]:
joined_reviews = joined.merge(
    coffee_gdf[['id', 'general_review_count']],
    on='id',
    how='left',
    validate='many_to_one'
)

print(
    joined_reviews.loc[
        joined_reviews["id"].notna(),
        ["candidate_id", "id", "general_review_count"]
    ].head(10)
)



    candidate_id                 id  general_review_count
23            23  70000001109683356                 386.0
24            23  70000001076927300                 174.0
25            24  70000001110553670                 166.0
26            25  70000001112756823                  67.0
27            25  70000001118455588                  19.0
37            35  70000001110482061                 267.0
38            35  70000001078539446                  49.0
39            36  70000001110482061                 267.0
40            36  70000001078539446                  49.0
41            37  70000001109683356                 386.0


In [14]:
'''
groupby("candidate_id") — объединяем строки, относящиеся к одной локации.
["general_review_count"] — выбираем столбец с отзывами.
.sum(min_count=1) — суммируем известные значения, но возвращаем NaN, если все значения отсутствуют.
'''

reviews_sum = (
    joined_reviews.groupby('candidate_id')['general_review_count'].sum(min_count=1)
)

print(reviews_sum.head(40))


candidate_id
0        NaN
1        NaN
2        NaN
3        NaN
4        NaN
5        NaN
6        NaN
7        NaN
8        NaN
9        NaN
10       NaN
11       NaN
12       NaN
13       NaN
14       NaN
15       NaN
16       NaN
17       NaN
18       NaN
19       NaN
20       NaN
21       NaN
22       NaN
23     560.0
24     166.0
25      86.0
26       NaN
27       NaN
28       NaN
29       NaN
30       NaN
31       NaN
32       NaN
33       NaN
34       NaN
35     316.0
36     316.0
37     386.0
38    3544.0
39    3544.0
Name: general_review_count, dtype: float64


In [ ]:
master["dgis_coffee_reviews_sample_500m"] = (
    master["candidate_id"].map(reviews_sum)
)

# Если заведений нет, сумма отзывов равна 0
no_coffee = master["dgis_coffee_sample_500m"] == 0

master.loc[
    no_coffee, "dgis_coffee_reviews_sample_500m"
] = 0



In [16]:
col = "dgis_coffee_reviews_sample_500m"

print(master[col].describe())
print("Missing:", master[col].isna().sum())
print("Maximum:", master[col].max())

print(
    master.loc[
        master["candidate_id"].isin([23, 25, 35]),
        ["candidate_id", col]
    ]
)

count     2790.000000
mean       432.668459
std       1788.749629
min          0.000000
25%          0.000000
50%          0.000000
75%          5.000000
max      25277.000000
Name: dgis_coffee_reviews_sample_500m, dtype: float64
Missing: 32
Maximum: 25277.0
    candidate_id  dgis_coffee_reviews_sample_500m
23            23                            560.0
25            25                             86.0
35            35                            316.0


In [17]:
missing = master[
    master["dgis_coffee_reviews_sample_500m"].isna()
]

print("Missing rows:", len(missing))

print(
    missing[
        ["candidate_id", "dgis_coffee_sample_500m"]
    ].head(15)
)

print(
    "Missing with zero competitors:",
    (missing["dgis_coffee_sample_500m"] == 0).sum()
)

Missing rows: 32
      candidate_id  dgis_coffee_sample_500m
375            375                        1
397            397                        1
428            428                        1
431            431                        2
432            432                        2
448            448                        1
463            463                        1
464            464                        1
479            479                        2
483            483                        2
514            514                        1
515            515                        1
529            529                        1
1045          1045                        1
1093          1093                        4
Missing with zero competitors: 0


In [19]:
#joined отвечает на вопрос - какие заведения находятся в радиусе 500 метров от каждой потенциальной локации?
#merge делает соединение таблиц когда sjoin делает соединение по геометрии
#on='id', соединяем по id
#Например, в joined есть id = 101. Pandas ищет id = 101 в coffee_gdf и переносит рейтинг 4.8.
#how=left, соединяет значения к joined чтобы если рейтинга не будет то будет просто None и строка не удалится



joined_ratings = joined.merge(
    coffee_gdf[['id', 'general_rating']],
    on='id',
    how='left',
    validate='many_to_one'
)

'''
groupby("candidate_id") — собираем в одну группу все заведения, находящиеся около конкретного кандидата.
["general_rating"] — выбираем рейтинги этих заведений.
'''

rating_mean = joined_ratings.groupby('candidate_id')['general_rating'].mean()

#сопоставляет нашему кандидату среднее значения рейтинга
master["dgis_coffee_rating_mean_sample_500m"] = (
    master["candidate_id"].map(rating_mean)
)


col = "dgis_coffee_rating_mean_sample_500m"

print(master[col].describe())
print("Missing:", master[col].isna().sum())
print("Min:", master[col].min())
print("Max:", master[col].max())


count    761.000000
mean       4.588559
std        0.725072
min        0.000000
25%        4.605000
50%        4.800000
75%        4.900000
max        5.000000
Name: dgis_coffee_rating_mean_sample_500m, dtype: float64
Missing: 2061
Min: 0.0
Max: 5.0


In [20]:
print("Zero ratings:", (coffee_gdf["general_rating"] == 0).sum())

print(
    coffee_gdf.loc[
        coffee_gdf["general_rating"] == 0,
        ["id", "name", "general_rating", "general_review_count"]
    ].head(20).to_string(index=False)
)

Zero ratings: 5
               id                               name  general_rating  general_review_count
70000001105326358             Center coffee, кофейня             0.0                   1.0
70000001109975692 Infinity, ​аппарат по продаже кофе             0.0                   1.0
70000001114359656               Rina Coffee, кофейня             0.0                   1.0
70000001119017434                Naz Coffee, кофейня             0.0                   3.0
70000001119279182                    My drink waffle             0.0                   1.0


Для нашей модели разумно считать такие рейтинги неизвестными (NaN), сохранив исходные данные без изменений.

При этом количество отзывов оставляем как есть. Например, у Center Coffee один отзыв — это информация, даже если итогового рейтинга нет.

In [22]:
ratings_clean = coffee_gdf[['id', 'general_rating']].copy()

ratings_clean['general_rating'] = (
    ratings_clean['general_rating'].replace(0, float('nan'))
)

joined_ratings = joined.merge(
    ratings_clean,
    on='id',
    how='left',
    validate='many_to_one'
)

rating_mean = joined_ratings.groupby('candidate_id')['general_rating'].mean()

# Обновляем признак
master["dgis_coffee_rating_mean_sample_500m"] = (
    master["candidate_id"].map(rating_mean)
)

# Проверяем
col = "dgis_coffee_rating_mean_sample_500m"

print(master[col].describe())
print("Missing:", master[col].isna().sum())
print("Zero ratings:", (master[col] == 0).sum())

count    757.000000
mean       4.620632
std        0.631228
min        1.000000
25%        4.640000
50%        4.800000
75%        4.900000
max        5.000000
Name: dgis_coffee_rating_mean_sample_500m, dtype: float64
Missing: 2065
Zero ratings: 0


In [23]:
features_2gis = [
    "dgis_coffee_sample_500m",
    "dgis_coffee_reviews_sample_500m",
    "dgis_coffee_rating_mean_sample_500m"
]

print("Shape:", master.shape)
print("Unique IDs:", master["candidate_id"].is_unique)
print(master[features_2gis].isna().sum())

Shape: (2822, 45)
Unique IDs: True
dgis_coffee_sample_500m                   0
dgis_coffee_reviews_sample_500m          32
dgis_coffee_rating_mean_sample_500m    2065
dtype: int64


In [24]:
from pathlib import Path
import geopandas as gpd

path = Path("../data/processed/master_eda_v0_1.parquet")

master.to_parquet(path, index=False)

check = gpd.read_parquet(path)

print("Saved:", path.exists())
print("Shape:", check.shape)
print("CRS:", check.crs.to_epsg())
print("Unique IDs:", check["candidate_id"].is_unique)

Saved: True
Shape: (2822, 45)
CRS: 32643
Unique IDs: True
